# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible analysis for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

It details the analysis of protein abundance, modifications, and sequences in Homo sapiens samples, including fields such as protein accession, description, coverage, peptide counts, and molecular weight (MW), with calculated parameters like pI and normalized abundances.


In [ ]:
# Ensure `mlcroissant` is installed in the current environment
# Restart the runtime after first install (if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset using Croissant
dataset = mlc.Dataset(croissant_url)

# Access top level metadata
print(f"Dataset Name: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n\nPublished: {dataset.metadata.datePublished}\n\nLicense: {dataset.metadata.license}\n")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets and field IDs via the Croissant schema. For this dataset, let's enumerate the available record sets and list their fields.

In [ ]:
# List all record sets with their `@id`s and fields
print("Available record sets (by @id):\n")
for rs in dataset.metadata.recordSets:
    print(f"- {rs['@id']} : {rs.get('name', '')}")

    print("  Fields (by @id):")
    for field in rs['field']:
        if isinstance(field, dict):
            print(f"    - {field['@id']:30} (name: {field.get('name', '')}, type: {field.get('dataType', '')})")
        else:
            # In case the schema just gives @ids, resolve to full field dict for name/type
            fdef = next(f for f in dataset.metadata.fields if f['@id']==field)
            print(f"    - {fdef['@id']:30} (name: {fdef.get('name', '')}, type: {fdef.get('dataType', '')})")
    print()


## 3. Data Extraction
Extract records from a chosen record set into a DataFrame for analysis. Use the `@id` values from above. The primary data table is typically the main record set whose fields include biological/experimental values.

In [ ]:
# List the record set IDs
record_sets = [rs['@id'] for rs in dataset.metadata.recordSets]
print(f"Record sets: {record_sets}")

# Let's extract all dataframes for all record sets
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from record set: {record_set_id}")
    # records() yields a dict per row, with keys as field @id or name
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(f"Head for {record_set_id}:")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Let's perform common EDA steps:
- Select a numeric field (e.g., coverage, MW, peptide count) by referencing its field `@id`.
- Filter the DataFrame for significant entries.
- Normalize a numeric column.
- Optionally, group by a categorical field.

In [ ]:
# Choose the main record set for analysis (use the first in the list for demonstration)
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]

# List numeric field @ids. This usually includes columns for coverage, MW, peptide counts, etc.
print("All columns:", main_df.columns.tolist())

# Let's try a common field name for abundance or coverage, or fallback to MW/peptide count.
# The field @ids may be like 'cr:abundance', 'cr:coverage', 'cr:MW', etc.
candidate_fields = [c for c in main_df.columns if any(x in c.lower() for x in ["abundance","coverage","mw","peptide"])]
if candidate_fields:
    numeric_field_id = candidate_fields[0]
else:
    numeric_field_id = main_df.columns[0]  # fallback
print(f"Using numeric field: {numeric_field_id}")

# Filter entries above a threshold (e.g., abundance>10 or coverage>10)
threshold = 10
if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
else:
    # Try converting in case data was loaded as string
    filtered_df = main_df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, e.g. 'cr:sample' or a name field
group_candidates = [c for c in main_df.columns if any(x in c.lower() for x in ["sample","experiment","cell"]) and c != numeric_field_id]

if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by field: {group_field}")
    if group_field in filtered_df.columns:  # defensive for missing columns
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No categorical field for grouping found.")


## 5. Visualization
Visualize numeric data distributions and relationships with matplotlib/seaborn.
Below, we plot the distribution of the selected numeric field, before and after normalization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the (filtered, normalized) numeric field
plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
sns.histplot(main_df[numeric_field_id], kde=True, bins=30, color='gray')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")

plt.subplot(1,2,2)
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30, color='dodgerblue')
plt.title(f"Filtered & Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

# If we found a categorical grouping, show boxplot
if group_candidates and group_field in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"Distribution of {numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
This notebook demonstrates how to access, process, and visualize data from the FAIR² Croissant package using the `mlcroissant` library. All data elements are referenced by their Croissant schema `@id`, ensuring reproducibility and clarity for further data science tasks such as downstream statistical analysis, ML model development, or comparison across experiments.

Explore additional record sets or fields as needed for your own application.
